# D-limonene in *Fusarium proliferatum*: publisher-table audit

This notebook audits the checksum-pinned publisher workbook from Zhou et al. (2025). It checks RNA-seq sample counts (Table S3), published differential-expression entries (Table S4), and the adjusted-p KEGG lists associated with H3K9ac/H3K27ac downregulated genes (Table S5). The workbook contains author-computed statistics; this notebook does not reprocess PRJNA1234870 reads or establish a direct molecular target.

In [1]:
from pathlib import Path
import csv, hashlib, json, math
from openpyxl import load_workbook

source = Path("research/fusarium-proliferatum-dlimonene-supplement/data/12864_2025_12389_MOESM2_ESM.xlsx")
expected = "bfedef70fb595d62d0c8080b3dfe67f9d8b8291fcb43af2b5642ce6d89489212"
actual = hashlib.sha256(source.read_bytes()).hexdigest()
assert actual == expected, f"Publisher workbook hash changed: {actual}"
book = load_workbook(source, read_only=True, data_only=True)
print("SHA-256:", actual)
print("Sheets:", ", ".join(book.sheetnames))

SHA-256: bfedef70fb595d62d0c8080b3dfe67f9d8b8291fcb43af2b5642ce6d89489212
Sheets: Table S1, Tables S2, Table S3, Table S4, Table S5, Table S6


## Sample and DEG table structure

Table S3 identifies three untreated `CK` and three D-limonene libraries. Table S4 prints down- and upregulated records side by side; each side is parsed independently. The `FPRO_` identifier count is distinguished from all feature rows because the publisher summary may exclude a non-`FPRO_` RNA feature.

In [2]:
sample_rows = []
for row in list(book["Table S3"].values)[2:]:
    if not row[0]: continue
    sample_rows.append({"sample": str(row[0]), "total_reads": int(row[1]), "mapped_fraction": float(row[2])})
assert len(sample_rows) == 6
assert sum(row["sample"].startswith("CK-") for row in sample_rows) == 3
assert sum(row["sample"].startswith("D-Limonene-") for row in sample_rows) == 3

sheet = book["Table S4"]
header = list(sheet.iter_rows(min_row=3, max_row=3, values_only=True))[0]
assert tuple(header[:4]) == ("Gene names", "log2FoldChange", "pvalue", "padj")
assert tuple(header[6:10]) == ("Gene names", "log2FoldChange", "pvalue", "padj")
deg = []
for row in sheet.iter_rows(min_row=4, values_only=True):
    for direction, offset in (("down", 0), ("up", 6)):
        if not row[offset]: continue
        gene, fold, pvalue, padj = row[offset:offset+4]
        item = {"feature_id": str(gene), "direction": direction, "log2_fold_change": float(fold),
                "p_value": float(pvalue), "adjusted_p_value": float(padj)}
        assert all(math.isfinite(item[k]) for k in ("log2_fold_change", "p_value", "adjusted_p_value"))
        assert 0 <= item["p_value"] <= 1 and 0 <= item["adjusted_p_value"] <= 1
        assert (item["log2_fold_change"] < 0) == (direction == "down")
        deg.append(item)
assert len({row["feature_id"] for row in deg}) == len(deg)
print("Samples:", len(sample_rows), "mapped fraction range:", min(x["mapped_fraction"] for x in sample_rows), max(x["mapped_fraction"] for x in sample_rows))
print("DEG table entries:", len(deg), "down:", sum(x["direction"] == "down" for x in deg), "up:", sum(x["direction"] == "up" for x in deg))

Samples: 6 mapped fraction range: 0.9013 0.9422
DEG table entries: 2375 down: 1240 up: 1135


## Published threshold and KEGG-list audit

All Table S4 entries are checked against adjusted p < 0.05 and absolute log2 fold change ≥ 1. Table S5 is a pathway list for genes the authors associate with reduced histone acetylation. Its adjusted p values are read as printed; one marked pathway is not a direct target or a causal mediation estimate.

In [3]:
pathways = []
s5 = book["Table S5"]
for row in s5.iter_rows(min_row=4, values_only=True):
    for mark, offset in (("H3K9ac", 0), ("H3K27ac", 8)):
        if not row[offset]: continue
        pathway, description, pvalue, padj = row[offset:offset+4]
        pathways.append({"mark": mark, "kegg_id": str(pathway), "description": str(description),
                         "p_value": float(pvalue), "adjusted_p_value": float(padj)})
summary = {
    "source_sha256": actual, "sample_count": len(sample_rows),
    "untreated_samples": sum(x["sample"].startswith("CK-") for x in sample_rows),
    "treated_samples": sum(x["sample"].startswith("D-Limonene-") for x in sample_rows),
    "mapped_fraction_min": min(x["mapped_fraction"] for x in sample_rows),
    "mapped_fraction_max": max(x["mapped_fraction"] for x in sample_rows),
    "published_deg_feature_rows": len(deg),
    "published_down_rows": sum(x["direction"] == "down" for x in deg),
    "published_up_rows": sum(x["direction"] == "up" for x in deg),
    "published_up_fpro_gene_rows": sum(x["direction"] == "up" and x["feature_id"].startswith("FPRO_") for x in deg),
    "non_fpro_feature_ids": sorted(x["feature_id"] for x in deg if not x["feature_id"].startswith("FPRO_")),
    "non_fpro_up_ids": sorted(x["feature_id"] for x in deg if x["direction"] == "up" and not x["feature_id"].startswith("FPRO_")),
    "non_fpro_down_ids": sorted(x["feature_id"] for x in deg if x["direction"] == "down" and not x["feature_id"].startswith("FPRO_")),
    "rows_failing_reported_thresholds": sum(x["adjusted_p_value"] >= 0.05 or abs(x["log2_fold_change"]) < 1 for x in deg),
    "h3k9ac_pathway_rows": sum(x["mark"] == "H3K9ac" for x in pathways),
    "h3k27ac_pathway_rows": sum(x["mark"] == "H3K27ac" for x in pathways),
    "h3k9ac_adjusted_p_below_0_05": sum(x["mark"] == "H3K9ac" and x["adjusted_p_value"] < 0.05 for x in pathways),
    "h3k27ac_adjusted_p_below_0_05": sum(x["mark"] == "H3K27ac" and x["adjusted_p_value"] < 0.05 for x in pathways),
    "significant_h3k9ac_pathways": [x["kegg_id"] for x in pathways if x["mark"] == "H3K9ac" and x["adjusted_p_value"] < 0.05],
    "effect_class": "author-processed differential-expression and pathway tables; audit only",
}
out = Path("research/fusarium-proliferatum-dlimonene-supplement/results")
out.mkdir(parents=True, exist_ok=True)
(out / "analysis_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
for name, rows in (("sample_qc.tsv", sample_rows), ("published_degs.tsv", deg), ("acetylation_pathways.tsv", pathways)):
    with (out / name).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), delimiter="\t")
        writer.writeheader(); writer.writerows(rows)
print(json.dumps(summary, indent=2))

{
  "source_sha256": "bfedef70fb595d62d0c8080b3dfe67f9d8b8291fcb43af2b5642ce6d89489212",
  "sample_count": 6,
  "untreated_samples": 3,
  "treated_samples": 3,
  "mapped_fraction_min": 0.9013,
  "mapped_fraction_max": 0.9422,
  "published_deg_feature_rows": 2375,
  "published_down_rows": 1240,
  "published_up_rows": 1135,
  "published_up_fpro_gene_rows": 1134,
  "non_fpro_feature_ids": [
    "rrna13",
    "rrna28",
    "rrna38",
    "rrna47",
    "rrna54"
  ],
  "non_fpro_up_ids": [
    "rrna13"
  ],
  "non_fpro_down_ids": [
    "rrna28",
    "rrna38",
    "rrna47",
    "rrna54"
  ],
  "rows_failing_reported_thresholds": 0,
  "h3k9ac_pathway_rows": 268,
  "h3k27ac_pathway_rows": 184,
  "h3k9ac_adjusted_p_below_0_05": 1,
  "h3k27ac_adjusted_p_below_0_05": 0,
  "significant_h3k9ac_pathways": [
    "ko03010"
  ],
  "effect_class": "author-processed differential-expression and pathway tables; audit only"
}


## Interpretation

The article reports 1,134 upregulated and 1,240 downregulated genes. Table S4 contains 1,135 upregulated feature rows and 1,240 downregulated rows; `rrna13` is the sole non-`FPRO_` upregulated entry, leaving 1,134 upregulated `FPRO_` entries. Four downregulated entries also have `rrna` identifiers, yet the article retains the full down count. Thus the one-row up discrepancy is visible, but a consistent exclusion rule is not documented in the workbook. One H3K9ac-associated KEGG term, ribosome (`ko03010`), has adjusted p < 0.05 in Table S5; none of the H3K27ac-associated terms does. These are within-study author-processed results at a strongly growth-inhibiting 200 mM, 24-hour exposure. They cannot establish direct limonene binding, an acetylation-mediated causal chain, or cross-study replication with the much lower-dose Penicillium biotransformation experiment.